# FireBridge 5/5: Export for the web app
Turns notebook 4's results into small JSON files the Next.js app reads from `public/data/`. No backend needed.

**Input:** `FireBridge/data/harmonized/` (notebook 4) and `FireBridge/data/regions.csv` (notebook 1).  
**Output:** `FireBridge/web_data/` on Drive + a zip to download.

| File | What the app uses it for |
|---|---|
| `regions.json` | Region picker: name, bbox, fire type, known event |
| `calendar_<region>.json` | Calendar and yearly chart: every month 2000–2024 with raw MODIS, real VIIRS, harmonized value and band |
| `model.json` | Test scores, 2012 jump check, anomalies |
| `cells/<region>/index.json`, `cells/<region>/<year>.json` | Map: 10 km cells and their monthly values |

Runs on CPU in under a minute.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Functions
Run once per session: `loadInputs`, `exportRegions`, `exportCalendars`, `exportReport`, `exportCells`, `exportAll`.

In [ ]:
import os, json, math, ast, shutil
import numpy as np
import pandas as pd

DATA_DIR = '/content/drive/MyDrive/FireBridge/data'
HARMONIZED_DIR = f'{DATA_DIR}/harmonized'          # written by notebook 4
WEB_DIR = '/content/drive/MyDrive/FireBridge/web_data'   # copy this folder to the repo's public/data/

_R, _K0 = 6371007.181, math.cos(math.radians(30))   # EPSG:6933, same grid as notebook 4
CELL_MIN = 0.5    # a cell-month is exported for the map when any value reaches this many detections


def _r(x, d=1):
    """Round for JSON; None for missing."""
    if x is None or (isinstance(x, float) and math.isnan(x)) or pd.isna(x):
        return None
    return round(float(x), d)


def _write(rel, obj):
    p = f'{WEB_DIR}/{rel}'
    os.makedirs(os.path.dirname(p), exist_ok=True)
    with open(p, 'w') as f:
        json.dump(obj, f, separators=(',', ':'))
    return os.path.getsize(p)


def _minimalReport(monthly, top=8):
    """Jump check + anomalies from the monthly table, for when notebook 4's report.json is missing."""
    m = monthly.copy(); m['month'] = pd.to_datetime(m['month'])
    jump, anom = [], []
    for a, d in m.groupby('area'):
        y = d.groupby(d.month.dt.year)[['modis_count', 'viirs', 'harmonized']].sum(min_count=1)
        b, af = y.loc[2009:2011].mean(), y.loc[2013:2015].mean()
        jump.append({'area': a, 'raw jump ×': af['viirs'] / max(b['modis_count'], 1e-9),
                     'real change ×': af['modis_count'] / max(b['modis_count'], 1e-9),
                     'harmonized jump ×': af['viirs'] / max(b['harmonized'], 1e-9)})
        d = d.copy(); d['mo'] = d.month.dt.month
        st = d.groupby('mo')['calendar'].agg(['mean', 'std'])
        d = d.join(st, on='mo'); d['z'] = (d['calendar'] - d['mean']) / d['std'].replace(0, np.nan)
        for _, r in d.nlargest(top, 'z').iterrows():
            anom.append({'area': a, 'month': r.month.strftime('%Y-%m'), 'calendar': float(r.calendar),
                         'source': r.source, 'z': float(r.z)})
    return {'grid_km': 10, 'years': [int(m.month.dt.year.min()), int(m.month.dt.year.max())], 'viirs_from': 2013,
            'split': {'train_until': 2020, 'val_years': [2021], 'test_from': 2022}, 'features': [],
            'model': 'Ratio × learned correction (main)', 'members': 5,
            'generated_at': pd.Timestamp.now(tz='UTC').isoformat(timespec='seconds'),
            'validation': [], 'test_overall': [], 'test_by_region': [], 'leave_one_region_out': [],
            'jump_check': jump, 'anomalies': anom}


def loadInputs():
    """Harmonized tables + report from notebook 4, region metadata from notebook 1."""
    monthly = pd.read_parquet(f'{HARMONIZED_DIR}/harmonized_monthly.parquet')
    cells = pd.read_parquet(f'{HARMONIZED_DIR}/harmonized_cells.parquet')
    rp = f'{HARMONIZED_DIR}/report.json'
    if os.path.exists(rp):
        with open(rp) as f:
            report = json.load(f)
    else:
        print('⚠️  report.json not found: it is written by the latest notebook 4 (its last cell, "10. Save").\n'
              '    Exporting without test scores; the jump check and anomalies are recomputed here.\n'
              '    For the full report, re-run notebook 4 from the latest version on GitHub.')
        report = _minimalReport(monthly)
    regions = pd.read_csv(f'{DATA_DIR}/regions.csv')
    regions['bbox'] = regions['bbox'].apply(ast.literal_eval)
    regions = regions[regions.area.isin(monthly.area.unique())].reset_index(drop=True)
    print(f'{len(monthly):,} region-months · {len(cells):,} cell-months · regions: {", ".join(regions.area)}')
    return monthly, cells, report, regions


def exportRegions(regions):
    out = []
    for _, r in regions.iterrows():
        lat0, lat1, lon0, lon1 = r['bbox']
        span = max(lat1 - lat0, lon1 - lon0)
        out.append({
            'id': r['area'], 'name': r['area'].split('_', 1)[1].replace('_', ' ') if '_' in r['area'] else r['area'],
            'country': r['country'].replace('_', ' '), 'priority': r['priority'],
            'fireType': r['fire_type'], 'goal': r['goal'],
            'bbox': [lat0, lat1, lon0, lon1], 'center': [round((lat0 + lat1) / 2, 3), round((lon0 + lon1) / 2, 3)],
            'zoom': int(np.clip(round(9 - math.log2(max(span, 0.5))), 4, 9)),
            'hemisphere': 'S' if (lat0 + lat1) / 2 < 0 else 'N',
            'eventStart': r['focus_start'], 'eventEnd': r['focus_end'],
        })
    size = _write('regions.json', out)
    print(f'regions.json · {len(out)} regions · {size / 1e3:.1f} kB')
    return out


def exportCalendars(monthly):
    """calendar_<id>.json: every month 2000-2024 for one region."""
    total = 0
    for a, d in monthly.groupby('area'):
        d = d.sort_values('month')
        months = [{'month': m.strftime('%Y-%m'), 'modis': int(r.modis_count), 'terra': int(r.terra_count),
                   'aqua': int(r.aqua_count), 'viirs': _r(r.viirs, 0), 'harmonized': _r(r.harmonized),
                   'low': _r(r.low), 'high': _r(r.high), 'calendar': _r(r.calendar), 'source': r.source}
                  for m, r in zip(pd.to_datetime(d.month), d.itertuples())]
        total += _write(f'calendar_{a}.json', {'regionId': a, 'months': months})
    print(f'calendar_*.json · {monthly.area.nunique()} files · {total / 1e3:.0f} kB')


def _clean(obj):
    """JSON-friendly keys ('Total bias %' -> 'total_bias'), months as 'YYYY-MM', floats rounded."""
    import re
    if isinstance(obj, dict):
        return {re.sub(r'[^0-9a-zA-Z]+', '_', k).strip('_').lower(): _clean(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_clean(v) for v in obj]
    if isinstance(obj, float):
        return None if math.isnan(obj) else round(obj, 3)
    if isinstance(obj, str) and len(obj) >= 10 and obj[4] == '-' and obj[7] == '-' and obj[:4].isdigit() and 'T00:00:00' in obj:
        return obj[:7]
    return obj


def exportReport(report):
    size = _write('model.json', _clean(report))
    print(f'model.json · {size / 1e3:.1f} kB')


def exportCells(cells, grid_km=10):
    """Map data per region: cells/<id>/index.json (cell rectangles) + cells/<id>/<year>.json (values by month).

    Only cell-months where MODIS, VIIRS or the harmonized value reaches CELL_MIN are written, to keep files small.
    Values per entry: [cell index, harmonized, modis, viirs or null].
    """
    g = grid_km * 1000
    total, nfiles = 0, 0
    for a, d in cells.groupby('area'):
        idx = d[['gx', 'gy']].drop_duplicates().sort_values(['gx', 'gy']).reset_index(drop=True)
        x0, y0 = idx.gx * g, idx.gy * g
        lon = lambda x: np.degrees(x / (_R * _K0))
        lat = lambda y: np.degrees(np.arcsin(np.clip(y * _K0 / _R, -1, 1)))
        bounds = np.column_stack([lat(y0), lon(x0), lat(y0 + g), lon(x0 + g)]).round(4).tolist()
        total += _write(f'cells/{a}/index.json', {'regionId': a, 'gridKm': grid_km, 'cells': bounds})
        key = {(gx, gy): i for i, (gx, gy) in enumerate(zip(idx.gx, idx.gy))}
        d = d[(d.harmonized >= CELL_MIN) | (d.modis_count >= 1) | (d.viirs_count.fillna(0) >= 1)].copy()
        d['ci'] = [key[(gx, gy)] for gx, gy in zip(d.gx, d.gy)]
        d['ym'] = pd.to_datetime(d.month)
        for y, dy in d.groupby(d.ym.dt.year):
            months = {}
            for m, dm in dy.groupby(dy.ym.dt.strftime('%m')):
                months[m] = [[int(c), _r(h), int(mo), _r(v, 0)] for c, h, mo, v in
                             zip(dm.ci, dm.harmonized, dm.modis_count, dm.viirs_count)]
            total += _write(f'cells/{a}/{y}.json', {'regionId': a, 'year': int(y), 'months': months}); nfiles += 1
    print(f'cells/ · {cells.area.nunique()} regions · {nfiles} year files · {total / 1e6:.1f} MB')


def exportAll():
    if os.path.exists(WEB_DIR):
        shutil.rmtree(WEB_DIR)
    monthly, cells, report, regions = loadInputs()
    exportRegions(regions)
    exportCalendars(monthly)
    exportReport(report)
    exportCells(cells, report.get('grid_km', 10))
    zip_path = shutil.make_archive('/content/firebridge_web_data', 'zip', WEB_DIR)
    print(f'\nAll files in {WEB_DIR}\nZip for the repo: {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)')
    return zip_path

## Export everything

In [ ]:
zip_path = exportAll()

## Download the zip
Then either:
- unzip it into the repo's `public/data/` folder (replace what's there) and commit, or
- send the zip to Claude to commit it.

In [ ]:
from google.colab import files
files.download(zip_path)

## Quick look

In [ ]:
import json
with open(f'{WEB_DIR}/model.json') as f:
    r = json.load(f)
print('Model:', r['model'])
print('2012 check:')
for j in r['jump_check']:
    print(f"  {j['area']:32s} raw {j['raw_jump']:5.2f}x · real {j['real_change']:5.2f}x · harmonized {j['harmonized_jump']:5.2f}x")